In [4]:
%pip install pyarrow


   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   - -------------------------------------- 1.0/27.9 MB 12.7 MB/s eta 0:00:03
   ----- ---------------------------------- 3.9/27.9 MB 13.8 MB/s eta 0:00:02
   ---------- ----------------------------- 7.6/27.9 MB 14.7 MB/s eta 0:00:02
   ---------------- ----------------------- 11.3/27.9 MB 15.7 MB/s eta 0:00:02
   --------------------- ------------------ 14.7/27.9 MB 15.9 MB/s eta 0:00:01
   -------------------------- ------------- 18.6/27.9 MB 16.3 MB/s eta 0:00:01
   ------------------------------- -------- 22.3/27.9 MB 16.6 MB/s eta 0:00:01
   ------------------------------------- -- 26.0/27.9 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------  27.8/27.9 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 27.9/27.9 MB 14.6 MB/s  0:00:02



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pyarrow
print(pyarrow.__version__)

25.0.0


In [2]:
"""
elsevier_text_extraction.py

Processes already-sectioned Elsevier .txt exports (TITLE / AUTHORS /
ABSTRACT / INTRODUCTION / MATERIALS AND METHODS / ... / RESULTS / ... /
DISCUSSION / CONCLUSION) into the same chunk format used by the PDF
pipeline, so both sources merge cleanly in the next step.

Your Elsevier filenames are already DOI-safe (e.g.
10-1016_j-afres-2025-101009.txt), matching the project's existing
convention of replacing '.' with '-' and '/' with '_' in DOIs. That
safe form is used directly as the identifier, no reversal needed.

Output: chunks_elsevier.parquet, one row per chunk, columns:
    id, doi, section, chunk_index, chunk_text, source_type ("elsevier_txt")
"""

import os
import pandas as pd

from chunk_utils import build_chunk_records

ELSEVIER_FOLDER = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\DOWNLOAD ARTICLES\Elsevier_TXT_files"
OUTPUT_PARQUET = "chunks_elsevier.parquet"


def run():
    files = [f for f in sorted(os.listdir(ELSEVIER_FOLDER)) if f.endswith(".txt")]
    print(f"Found {len(files)} Elsevier txt files in {ELSEVIER_FOLDER}/")

    all_records = []
    for i, fname in enumerate(files, 1):
        path = os.path.join(ELSEVIER_FOLDER, fname)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        doi_safe = os.path.splitext(fname)[0]  # already DOI-safe, per project convention
        records = build_chunk_records(doi_safe, text)
        for r in records:
            r["source_type"] = "elsevier_txt"
        all_records.extend(records)
        print(f"[{i}/{len(files)}] {fname}  chunks={len(records)}")

    df = pd.DataFrame(all_records)
    df.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"\nSaved {len(df)} chunks from {len(files)} Elsevier articles to {OUTPUT_PARQUET}")


if __name__ == "__main__":
    run()

Found 112 Elsevier txt files in C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\DOWNLOAD ARTICLES\Elsevier_TXT_files/
[1/112] 10-1016_S0308-8146(02)00555-1.txt  chunks=4
[2/112] 10-1016_S0308-8146(03)00203-6.txt  chunks=4
[3/112] 10-1016_j-afres-2025-101009.txt  chunks=9
[4/112] 10-1016_j-fbio-2023-102761.txt  chunks=6
[5/112] 10-1016_j-fhfh-2021-100015.txt  chunks=7
[6/112] 10-1016_j-fochms-2026-100388.txt  chunks=8
[7/112] 10-1016_j-fochx-2024-101957.txt  chunks=7
[8/112] 10-1016_j-fochx-2025-102898.txt  chunks=6
[9/112] 10-1016_j-fochx-2026-103624.txt  chunks=8
[10/112] 10-1016_j-fochx-2026-104017.txt  chunks=6
[11/112] 10-1016_j-foodchem-2003-09-036.txt  chunks=6
[12/112] 10-1016_j-foodchem-2004-12-041.txt  chunks=4
[13/112] 10-1016_j-foodchem-2006-01-011.txt  chunks=0
[14/112] 10-1016_j-foodchem-2006-08-012.txt  chunks=5
[15/112] 10-1016_j-foodchem-2008-09-025.txt  chunks=5
[16/112] 10-1016_j-foodchem-2009-01-011.txt  chunks=6
[17/112] 10

# READ THE PARQUET FILE

In [3]:
import pandas as pd

df = pd.read_parquet("chunks_elsevier.parquet")

with open("chunks_elsevier_preview.txt", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(f"===== {row['id']} =====\n")
        f.write(row["chunk_text"])
        f.write("\n\n")

print(f"Wrote {len(df)} chunks to chunks_elsevier_preview.txt")

Wrote 745 chunks to chunks_elsevier_preview.txt
